<h1>Sample Problem 6-4</h1>
5/5/26
<br>
Compound Pendulum simulated in opensim. <br>
pin.calcReactionOnParentExpressedInGround(s)


In [68]:
import opensim as osim
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.fft import fft, fftfreq

In [2]:

table = osim.TimeSeriesTable('data/pendulum_coordinates.sto')

# Column labels and time vector
labels = [table.getColumnLabel(i) for i in range(table.getNumColumns())]
time   = list(table.getIndependentColumn())

# Pull data matrix
data = table.getMatrix().to_numpy()   # shape (nRows, nCols)

df = pd.DataFrame(data, columns=labels)
df.insert(0, 'time', time)

In [4]:
df.head()

,time,theta_rad,dtheta_rad_s
0,0.00,1.570796,0.000000
1,0.01,1.569387,-0.281816
2,0.02,1.565160,-0.563643
3,0.03,1.558114,-0.845465
4,0.04,1.548250,-1.127236


In [72]:
plt.figure()
plt.plot(df["time"], df["theta_rad"])
plt.show()
N = len(df["theta_rad"])
fs = 100
xf = fftfreq(N, 1 / fs)   # frequency axis
spec = (2/N) * np.abs(fft(df["theta_rad"]))[:N//2]
plt.figure()
plt.plot(xf[:N//2], spec)
plt.show()

In [74]:
freq = xf[np.argmax(spec)]
freq * 2*np.pi

5.016515215313043

In [17]:
key_idx = np.flatnonzero(df["theta_rad"] < np.deg2rad(30))[0]
print(f"Pendulum first reaches {np.rad2deg(df['theta_rad'].iloc[key_idx]):.2f} degrees: {df['time'].iloc[key_idx]} s")
print(f"Velocity at that time: {df['dtheta_rad_s'].iloc[key_idx-1]**2:.2f} rad/s")

Pendulum first reaches 29.12 degrees: 0.28 s
Velocity at that time: 47.18 rad/s


In [46]:
model = osim.Model('compound_pendulum.osim')

# Add a StatesTrajectoryReporter before simulating
reporter = osim.StatesTrajectoryReporter()
reporter.setName('states_reporter')
reporter.set_report_time_interval(0.01)
model.addComponent(reporter)

state = model.initSystem()
# ... set initial conditions, run manager ...
pin   = model.getJointSet().get('pivot_joint'); 
coord = pin.updCoordinate()

coord.setValue(state, np.pi/2)
coord.setSpeedValue(state, 0.0)

manager = osim.Manager(model);
manager.setIntegratorAccuracy(1e-5)
manager.initialize(state);
finalState = manager.integrate(5.0)


# Post-hoc analysis
states_traj = reporter.getStates()

records = []
for i in range(states_traj.getSize()):
    s = states_traj[i]
    
    # realizeAcceleration is REQUIRED before calcReactionOnParentExpressedInGround
    model.realizeAcceleration(s)
    
    # Returns a SpatialVec: [0] = moment (Vec3), [1] = force (Vec3)
    reaction = pin.calcReactionOnParentExpressedInGround(s)
    force  = reaction.get(1)   # Vec3
    moment = reaction.get(0)   # Vec3

    records.append({
        'time'  : s.getTime(),
        'Fx': force[0],  'Fy': force[1],  'Fz': force[2],
        'Mx': moment[0], 'My': moment[1], 'Mz': moment[2],
    })

df2 = pd.DataFrame(records)

In [56]:
df_force = df2[["time", "Fx", "Fy"]].copy()
df_force["F_rxn"]= np.sqrt(df_force["Fx"]**2 + df_force["Fy"]**2)
df_force.loc[:, "theta_rad"] = df["theta_rad"]


In [65]:
df_force["F_rxn"][key_idx]
Fn = df_force["Fy"][key_idx]*np.cos(df["theta_rad"].iloc[key_idx]) - df_force["Fx"][key_idx]*np.sin(df["theta_rad"].iloc[key_idx])
Ft = df_force["Fy"]*np.sin(df["theta_rad"]) + df_force["Fx"]*np.cos(df["theta_rad"])
Ft[key_idx]

-10.058674359391901

In [50]:
inds = df2["time"].between(0, 0.3)
plt.figure()
plt.plot(df["theta_rad"][inds], df2["Fy"][inds])
plt.show()

In [40]:
7.5*0.25*df["dtheta_rad_s"].iloc[key_idx-1]**2 - 9.81*0.25*np.cos(df["theta_rad"].iloc[key_idx-1])

86.40673990625406

In [39]:
df.iloc[key_idx] 

time            0.280000
theta_rad       0.508232
dtheta_rad_s   -7.012261
Name: 28, dtype: float64

In [25]:
df2["time"].between(0.5, 1)

0      False
1      False
2      False
3      False
4      False
       ...  
496    False
497    False
498    False
499    False
500    False
Name: time, Length: 501, dtype: bool

In [32]:
%matplotlib qt

In [36]:
inds = df2["time"].between(0.1, 0.3)
plt.figure()
plt.plot(df2["time"][inds], df2["Fy"][inds])
plt.plot(df2["time"][inds], df2["Fx"][inds])
plt.legend(['Fy', 'Fx'])
ax = plt.gca().twinx()
ax.plot(df2["time"][inds], np.rad2deg(df["theta_rad"][inds]), linestyle='--', color='gray')
plt.grid()
plt.show()